# DATA SPLITTING

In [ ]:
"""
Splits the preprocessed data into six sets:

LLM-annotated data:
  - train                 : LLM-annotated data for model training
  - val                   : LLM-annotated data for hyperparameter tuning
  - test_llm              : LLM-annotated held-out test set

Manual annotations:
  - test_manual_500   : Initial round of manual coding (500 articles)
  - train_manual_1800 : Verified predictions for training (80% of 2,000)
  - test_manual_200  : Verified predictions for testing (20% of 2,000)

All splits are stratified to preserve label distribution.
"""

In [1]:
import pandas as pd
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

In [4]:
# define paths

IN_LLM              = "/Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/processed/baseline/cleaned/llm_18k_clean.parquet"
IN_MANUAL_500       = "/Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/processed/test_manual_500.parquet"
IN_MANUAL_2000      = "/Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/processed/cycle1/cleaned/manual_2000_clean.parquet"
OUT_DIR             = "/Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/processed/"

LABEL_COLS  = [f"label_{i}" for i in range(8)]

In [5]:
# load data
llm_df = pd.read_parquet(IN_LLM)
manual_500_df = pd.read_parquet(IN_MANUAL_500)
manual_2000_df = pd.read_parquet(IN_MANUAL_2000)

In [6]:
# exclude manually coded articles from the LLM pool to prevent any overlap
# between the manual test sets and the LLM train/val/test splits.
manual_keys = set(manual_500_df["ArticleKey"]) | set(manual_2000_df["ArticleKey"])
llm_df = llm_df[~llm_df["ArticleKey"].isin(manual_keys)].reset_index(drop=True)

print(f"LLM pool (after removing manual articles): {len(llm_df)}")
print(f"Initial manual test set:                   {len(manual_500_df)}")
print(f"Verified manual set (to be split):         {len(manual_2000_df)}")

LLM pool (after removing manual articles): 17477
Initial manual test set:                   500
Verified manual set (to be split):         500


In [9]:
# split llm data into train and val
msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
for train_idx, test_idx in msss.split(llm_df, llm_df[LABEL_COLS]):
    train_df = llm_df.iloc[train_idx]
    val_df  = llm_df.iloc[test_idx]

print(f"\nLLM data split:")
print(f"  Train: {len(train_df)} ({100 * len(train_df) / len(llm_df):.1f}%)")
print(f"  Test:  {len(val_df)} ({100 * len(val_df) / len(llm_df):.1f}%)")


LLM data split:
  Train: 14850 (85.0%)
  Test:  2627 (15.0%)


In [8]:
# split verified manual data into train (80%) and test (20%)
msss_verified = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
for train_idx, test_idx in msss_verified.split(manual_2000_df, manual_2000_df[LABEL_COLS]):
    train_verified_df = manual_2000_df.iloc[train_idx]
    test_verified_df  = manual_2000_df.iloc[test_idx]

print(f"\nVerified manual data split:")
print(f"  Train: {len(train_verified_df)} ({100 * len(train_verified_df) / len(manual_2000_df):.1f}%)")
print(f"  Test:  {len(test_verified_df)} ({100 * len(test_verified_df) / len(manual_2000_df):.1f}%)")


Verified manual data split:
  Train: 451 (90.2%)
  Test:  49 (9.8%)


In [ ]:
# check label distribution
print("\nLabel distributions (mean per label):")
for name, split in [("Train (LLM)",              train_df),
                    ("Val (LLM)",                val_df),
                    ("Test LLM",                 test_df),
                    ("Test Manual 500",      manual_500_df),
                    ("Train Manual 2000",    train_verified_df),
                    ("Test Manual 2000",     test_verified_df)]:
    print(f"\n  {name} (n={len(split)}):")
    print(split[LABEL_COLS].mean().round(3))


Label distributions (mean per label):

  Train (LLM) (n=9890):
label_0    0.049
label_1    0.161
label_2    0.026
label_3    0.162
label_4    0.822
label_5    0.019
label_6    0.030
label_7    0.044
dtype: float64

  Val (LLM) (n=2488):
label_0    0.049
label_1    0.160
label_2    0.025
label_3    0.161
label_4    0.818
label_5    0.018
label_6    0.030
label_7    0.044
dtype: float64

  Test LLM (n=3097):
label_0    0.049
label_1    0.160
label_2    0.026
label_3    0.162
label_4    0.821
label_5    0.019
label_6    0.030
label_7    0.044
dtype: float64

  Test Manual Initial (n=500):
label_0    0.902
label_1    0.002
label_2    0.002
label_3    0.002
label_4    0.082
label_5    0.008
label_6    0.012
label_7    0.010
dtype: float64

  Train Manual Verified (n=1800):
label_0    0.950
label_1    0.001
label_2    0.001
label_3    0.002
label_4    0.045
label_5    0.002
label_6    0.004
label_7    0.003
dtype: float64

  Test Manual Verified (n=202):
label_0    0.946
label_1    0.000
la

In [ ]:
# check overlap
splits = {
    "train_llm":            set(train_df["ArticleKey"]),
    "val_llm":              set(val_df["ArticleKey"]),
    "test_manual_500":  set(manual_500_df["ArticleKey"]),
    "train_manual_1800": set(train_verified_df["ArticleKey"]),
    "test_manual_200":  set(test_verified_df["ArticleKey"]),
}
print("\nArticleKey overlap between splits:")
print("\nLLM splits:")
for a, b in [("train_llm", "val_llm"), ("train_llm", "test_llm"), ("val_llm", "test_llm")]:
    print(f"  {a} ∩ {b}: {len(splits[a] & splits[b])}")

print("\nManual verified splits:")
print(f"  train_manual_1800 ∩ test_manual_200: {len(splits['train_manual_1800'] & splits['test_manual_200'])}")

print("\nCross-dataset overlaps:")
for llm_split in ["train_llm", "val_llm", "test_llm"]:
    for manual_split in ["test_manual_500", "train_manual_1800", "test_manual_200"]:
        overlap = len(splits[llm_split] & splits[manual_split])
        if overlap > 0:
            print(f"{llm_split} ∩ {manual_split}: {overlap}")
print(f"  test_manual_500 ∩ train_manual_1800: {len(splits['test_manual_500'] & splits['train_manual_1800'])}")
print(f"  test_manual_500 ∩ test_manual_200: {len(splits['test_manual_500'] & splits['test_manual_200'])}")


ArticleKey overlap between splits:

LLM splits:
  train_llm ∩ val_llm: 0
  train_llm ∩ test_llm: 0
  val_llm ∩ test_llm: 0

Manual verified splits:
  train_manual_verified ∩ test_manual_verified: 0

Cross-dataset overlaps:
  test_manual_initial ∩ train_manual_verified: 0
  test_manual_initial ∩ test_manual_verified: 0


In [ ]:
# save splits
train_df.to_parquet(            OUT_DIR + "train.parquet")
val_df.to_parquet(              OUT_DIR + "val.parquet")
manual_500_df.to_parquet(       OUT_DIR + "test_manual_500.parquet")
train_verified_df.to_parquet(   OUT_DIR + "train_manual_1800.parquet")
test_verified_df.to_parquet(    OUT_DIR + "test_manual_200.parquet")

print("\nAll six splits saved:")
print(f"  train (LLM):                {len(train_df)}")
print(f"  val (LLM):                  {len(val_df)}")
print(f"  test_llm:                   {len(test_df)}")
print(f"  test_manual_500:        {len(manual_500_df)}")
print(f"  train_manual_1800:      {len(train_verified_df)}")
print(f"  test_manual_200:       {len(test_verified_df)}")


All six splits saved:
  train (LLM):                9890
  val (LLM):                  2488
  test_llm:                   3097
  test_manual_initial:        500
  train_manual_verified:      1800
  test_manual_verified:       202
